# Hof_malanga_comparison — DiscEvolution vs Malanga et al. (2026)

**Goal.** Test whether the weak correlation between disc mass $M$ and stellar accretion rate $\dot M_*$ that Malanga et al. (2026) find for MHD-wind-driven discs (their Fig. 4) also appears in DiscEvolution's hybrid viscous + wind model. The population here is drawn from Malanga's inferred MHD-wind distributions (their Eq. 7 and Table 2, MHD-winds column) at $M_* = 1\,M_\odot$, with:
- fixed viscous $\alpha_{SS} = 10^{-4}$;
- $\psi = \alpha_{\rm tot}/\alpha_{SS} - 1$;
- $e_{\rm rad}$ mapped from Malanga's $f_M$ (their Eq. 6), via the steady-state wind relation and the code's $\lambda(e_{\rm rad},\psi)$ (Balogh et al. 2026, Eq. 16).

**Pipeline** (all in `example/StartHere/`):
- `sample_malanga_Hof.py` → `config/malanga/malanga_draws_Hof_N500_seed1.csv` (one row per disc: parameters + sampled age $t_{\rm age}$)
- `run_popsynth_malanga_Hof.sh` + `config/popsynth/DiscConfig_Hof_malanga.json` (grid 0.1–250 AU, 500 cells, 34 snapshots to 3 Myr)
- outputs: one `hof_malanga_w0_<id>_*.h5` per disc, next to a copy of the population CSV

**Units** (see `CLAUDE.md`):

| Quantity | Units |
|---|---|
| HDF5 `t` | yr |
| `time_snap` | Myr |
| `disk_Mass` | g (÷ `Msun`) |
| `disk_Mdot_star` | M$_\odot$/yr |
| `T` | K |
| `R` | AU |

**Notebook sections**
1. Run check (this first cell): completion, runtime, file size, and the gravitational-instability (GI) flag of the initial disc, i.e. Toomre $Q < 1.5$ (Toomre 1964; Balogh et al. 2026, Eq. 48), with the radius where the unstable region begins.
2. *(to come)* parameter comparisons: achieved $f_M$ vs target, $M/\dot M$ vs analytic, …
3. *(to come)* Fig. 4 reproduction: $\dot M$–$M$ at each disc's sampled age, with analytic $\omega = 0$ / $\omega = 0.4$ shadow populations and the Lupus subsample near $1\,M_\odot$.


In [ ]:
# =====================================================================================
# 1. Run check: one row per disc in DATA_DIR (from check_malanga_pilot_Hof.py, reduced)
# =====================================================================================
# Columns:
#   id          disc id (row of the population CSV; run_name = hof_malanga_w0_<id>)
#   R0_au       characteristic radius R0 [AU]                       (population CSV)
#   M0_Msun     initial disc mass M0 [Msun]                         (population CSV)
#   complete    HDF5 attr complete = True AND last snapshot time_snap[-1] reached T_END_MYR (Myr)
#   runtime_h   wall-clock run_end - run_start [h] (HDF5 attrs; includes set-up)
#   size_MB     file size [MB]
#   GI_flag     gravitational instability of the INITIAL disc: Toomre (1964) Q = c_s Omega / (pi G Sigma)
#               (Balogh et al. 2026, Eq. 48) evaluated on every grid cell; flagged if Q < 1.5 anywhere
#               (see disc_setup_Hof.gi_check_toomre)
#   GI_R_au     radius [AU] where the disc ENTERS the unstable region going outward (innermost crossing
#               of Q = 1.5); NaN if not flagged. (The outer edge is in HDF5 attr GI_R_out_au.)
#   GI_source   'attr' = read from the HDF5 attrs written by run_model_Hof.py; 'recomputed' = file
#               written before those attrs existed (e.g. pilot1), recomputed here with the SAME function
#               from the t = 0 snapshot: gas surface density Sigma_G[0] and temperature T[0].
# Nothing is written to disk. Parameter comparisons come in later cells.
import os, sys, glob, re
from datetime import datetime
import numpy as np
import pandas as pd
import h5py

# ---- where things are -------------------------------------------------------------
DATA_DIR = "/home/imontesd/projects/def-mbalogh/imontesd/output/DiscEvolution/popsynth/malanga/pilot1"
T_END_MYR = 3.0          # Myr, last snapshot of DiscConfig_Hof_malanga.json
T_END_RTOL = 1e-3        # relative tolerance for "reached the end" (dimensionless)

STARTHERE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))     # notebook lives in StartHere/notebooks/
if STARTHERE_DIR not in sys.path:
    sys.path.insert(0, STARTHERE_DIR)
from disc_setup_Hof import gi_check_toomre                            # Toomre Q GI check (Balogh+26 Eq. 48)
from DiscEvolution.constants import AU, Omega0, GasConst              # cm, s^-1, erg/mol/K

MU_EOS = 2.4             # mean molecular weight IrradiatedEOS uses (its default; make_eos does not pass mu)


def cs_omega_code_units(R_au, T_K, M_star):
    """Sound speed and Keplerian Omega in DiscEvolution code units (G = AU = Msun = 1), exactly as
    IrradiatedEOS / SimpleStar compute them:
        c_s = cs0 sqrt(T),  cs0 = (Omega0^-1 / AU) sqrt(GasConst / mu)     [AU per code-time unit]
        Omega_K = sqrt(M* / R^3)                                             [per code-time unit]
    R in AU, T in K, M* in Msun."""
    cs = (1 / Omega0 / AU) * np.sqrt(GasConst / MU_EOS) * np.sqrt(T_K)
    return cs, np.sqrt(M_star / R_au ** 3)


def run_row(path, draws):
    """Run-check row for one .h5 file (see column list above)."""
    disc_id = int(re.search(r"hof_malanga_w0_(\d+)_", os.path.basename(path)).group(1))
    d = draws.loc[disc_id]
    row = {"id": disc_id, "R0_au": d.R0_au, "M0_Msun": d.M0_Msun}
    with h5py.File(path, "r") as h:
        a = h.attrs
        t_last = float(h["time_snap"][-1]) if ("time_snap" in h and len(h["time_snap"])) else np.nan   # Myr
        row["complete"] = bool(a.get("complete", False)) and bool(t_last >= T_END_MYR * (1 - T_END_RTOL))
        row["runtime_h"] = np.nan
        if "run_start" in a and "run_end" in a:
            row["runtime_h"] = (datetime.fromisoformat(str(a["run_end"]))
                                - datetime.fromisoformat(str(a["run_start"]))).total_seconds() / 3600
        row["size_MB"] = os.path.getsize(path) / 1e6
        if "GI_flag" in a and "GI_R_in_au" in a:                     # written by run_model_Hof.py
            row["GI_flag"], row["GI_R_au"], row["GI_source"] = bool(a["GI_flag"]), float(a["GI_R_in_au"]), "attr"
        elif "Sigma_G" in h and len(h["Sigma_G"]) and np.isclose(h["time_snap"][0], 0.0):
            # recompute from the t = 0 snapshot (same criterion and function as the attrs)
            R = h["R"][:]                                                            # AU
            cs, Om = cs_omega_code_units(R, h["T"][0], d.Mstar_Msun)
            gi = gi_check_toomre(R, h["Sigma_G"][0], cs, Om)                         # Sigma_G[0] in g/cm^2
            row["GI_flag"], row["GI_R_au"], row["GI_source"] = gi["GI_flag"], gi["GI_R_in_au"], "recomputed"
        else:
            row["GI_flag"], row["GI_R_au"], row["GI_source"] = np.nan, np.nan, "unavailable"
    return row


csvs = glob.glob(os.path.join(DATA_DIR, "malanga_draws_Hof_*.csv"))
assert len(csvs) == 1, f"expected exactly one malanga_draws_Hof_*.csv in {DATA_DIR}, found {csvs}"
draws = pd.read_csv(csvs[0]).set_index("id")
files = sorted(glob.glob(os.path.join(DATA_DIR, "hof_malanga_w0_*.h5")))
assert files, f"no hof_malanga_w0_*.h5 files in {DATA_DIR}"

run_check = pd.DataFrame([run_row(f, draws) for f in files]).sort_values("id").reset_index(drop=True)
print(f"{len(run_check)} discs in {DATA_DIR}  |  complete: {int(run_check.complete.sum())}  |  "
      f"GI-flagged (Q < 1.5): {int(run_check.GI_flag.fillna(False).sum())}  |  runtime median "
      f"{run_check.runtime_h.median():.2f} h, max {run_check.runtime_h.max():.2f} h  |  "
      f"total size {run_check.size_MB.sum():.0f} MB")
run_check.style.format({"R0_au": "{:.3g}", "M0_Msun": "{:.3g}", "runtime_h": "{:.2f}", "size_MB": "{:.1f}",
                        "GI_R_au": "{:.3g}"}, na_rep="—")


## Next sections (to be added)
- **Parameter comparisons:**
  - achieved $f_M$, instantaneous and cumulative, vs `f_M_grid_expected`;
  - $M/\dot M$ at $t_{\rm age}$ vs the analytic $\omega = 0$ value $2\,t_{\rm acc,0}(1+f_M)$;
  - growth of the disc radius for compact discs.
- **Fig. 4 reproduction:** $\dot M$ vs $M$ at each disc's sampled age, compared with Malanga's analytic model ($\omega = 0$ and $0.4$, same draws) and with Lupus sources near $1\,M_\odot$. Flags to carry through: GI-flagged discs, and discs with $R_0 > 250$ AU, which extend beyond the grid.
